# Guía de Laboratorio N.° 2 — Librerías para Datos y Automatización
### Tecnologías Emergentes (ISO46B) · Semana 2 · Facultad de Ingeniería de Sistemas — UNCP

**Versión: guía de trabajo del estudiante.**

Continúa sobre el repositorio `tecemer-lab1` creado en la Semana 1. Ejecute este notebook desde la raíz de ese repositorio. Los comandos de instalación usan la ruta directa al entorno virtual (`.venv/bin/...`); en Windows, fuera del notebook, use `.venv\\Scripts\\...`.

## Requisitos previos
- Repositorio `tecemer-lab1` de la Semana 1, con `.venv` ya creado.
- Conexión a internet (API Open-Meteo, sin clave de acceso).

## Parte 1 — NumPy: fundamentos y computación vectorizada (60 min) · Nivel: Básico

### Paso 1.1 — Instalar NumPy

In [ ]:
!.venv/bin/pip install numpy
!.venv/bin/pip freeze > requirements.txt

### Paso 1.2 — Creación de arrays

In [ ]:
%%writefile numpy_demo.py
import numpy as np

a = np.array([1, 2, 3, 4, 5])
ceros = np.zeros((2, 3))
rango = np.arange(0, 10, 2)
espaciado = np.linspace(0, 1, 5)

print('Array base:', a)
print('Matriz de ceros:\n', ceros)
print('Rango:', rango)
print('Espaciado:', espaciado)


In [ ]:
!.venv/bin/python numpy_demo.py

### Paso 1.3 — Operaciones vectorizadas vs. bucles

In [ ]:
%%writefile -a numpy_demo.py
import time

n = 1_000_000
lista = list(range(n))
array = np.arange(n)

inicio = time.time()
resultado_lista = [x + 10 for x in lista]
print('Bucle for:', time.time() - inicio, 'segundos')

inicio = time.time()
resultado_array = array + 10
print('Vectorizado:', time.time() - inicio, 'segundos')


In [ ]:
!.venv/bin/python numpy_demo.py

### Paso 1.4 — Mini-reto: estadística descriptiva

In [ ]:
%%writefile -a numpy_demo.py
temperaturas = np.array([12.5, 14.0, 11.8, 13.2, 15.1, 10.9, 12.7])

print('Promedio:', temperaturas.mean())
print('Desviacion estandar:', temperaturas.std())
print('Minima:', temperaturas.min())
print('Maxima:', temperaturas.max())


In [ ]:
!.venv/bin/python numpy_demo.py

**Checkpoint de Parte 1:** las cuatro estadísticas deben imprimirse sin errores.

## Parte 2 — Consumo de APIs REST y manejo de JSON (90 min) · Nivel: Intermedio

### Paso 2.1 — Instalar requests (si no está) y repasar manejo de errores

In [ ]:
!.venv/bin/pip install requests

### Paso 2.2 — Consumir la API Open-Meteo para Huancayo

In [ ]:
%%writefile clima.py
import requests
import json

URL = "https://api.open-meteo.com/v1/forecast"
PARAMETROS = {
    "latitude": -12.07,      # Huancayo
    "longitude": -75.21,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/Lima",
    "forecast_days": 7,
}

try:
    respuesta = requests.get(URL, params=PARAMETROS, timeout=5)
    respuesta.raise_for_status()
    datos = respuesta.json()
except requests.exceptions.RequestException as error:
    raise SystemExit(f"No se pudo obtener el pronostico: {error}")

print(json.dumps(datos["daily"], indent=2, ensure_ascii=False))


In [ ]:
!.venv/bin/python clima.py

### Paso 2.3 — Ejecutar clima.py dentro del notebook para reutilizar la variable `datos`

In [ ]:
import requests, json

URL = "https://api.open-meteo.com/v1/forecast"
PARAMETROS = {
    "latitude": -12.07,
    "longitude": -75.21,
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/Lima",
    "forecast_days": 7,
}
respuesta = requests.get(URL, params=PARAMETROS, timeout=5)
respuesta.raise_for_status()
datos = respuesta.json()
print('OK, datos obtenidos.')

In [ ]:
print("Claves de primer nivel:", list(datos.keys()))
print('Claves de "daily":', list(datos["daily"].keys()))
print("Tipo de temperature_2m_max:", type(datos["daily"]["temperature_2m_max"]))
print("Primer valor de tiempo:", datos["daily"]["time"][0])


### Paso 2.4 — Guardar el JSON crudo y convertir a CSV

In [ ]:
with open("pronostico_huancayo.json", "w", encoding="utf-8") as archivo:
    json.dump(datos, archivo, ensure_ascii=False, indent=2)


In [ ]:
import csv

diario = datos["daily"]
filas = zip(
    diario["time"],
    diario["temperature_2m_max"],
    diario["temperature_2m_min"],
    diario["precipitation_sum"],
)

with open("pronostico_huancayo.csv", "w", newline="", encoding="utf-8") as archivo:
    escritor = csv.writer(archivo)
    escritor.writerow(["fecha", "temp_max", "temp_min", "precipitacion"])
    escritor.writerows(filas)


**Checkpoint de Parte 2:** verifique con `!ls pronostico_huancayo.*` que ambos archivos existen.

In [ ]:
!ls pronostico_huancayo.*

## Parte 3 — Pandas: de datos crudos a análisis, y consolidación con Git (90 min) · Nivel: Avanzado

### Paso 3.1 — Instalar Pandas y cargar el CSV

In [ ]:
!.venv/bin/pip install pandas
!.venv/bin/pip freeze > requirements.txt

In [ ]:
%%writefile analisis.py
import pandas as pd

df = pd.read_csv("pronostico_huancayo.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
print(df.head())
print(df.info())


In [ ]:
import pandas as pd

df = pd.read_csv("pronostico_huancayo.csv")
df["fecha"] = pd.to_datetime(df["fecha"])
print(df.head())
print(df.info())


### Paso 3.2 — Exploración y transformación

In [ ]:
%%writefile -a analisis.py
df["amplitud_termica"] = df["temp_max"] - df["temp_min"]
df["dia_lluvioso"] = df["precipitacion"] > 0
df["categoria"] = df["temp_max"].apply(
    lambda t: "calido" if t >= 20 else ("templado" if t >= 15 else "frio")
)
print(df)
print(df.describe())


In [ ]:
df["amplitud_termica"] = df["temp_max"] - df["temp_min"]
df["dia_lluvioso"] = df["precipitacion"] > 0
df["categoria"] = df["temp_max"].apply(
    lambda t: "calido" if t >= 20 else ("templado" if t >= 15 else "frio")
)
print(df)
print(df.describe())


### Paso 3.3 — Agregaciones con groupby

In [ ]:
%%writefile -a analisis.py
resumen = df.groupby("categoria").agg(
    dias=("categoria", "count"),
    temp_max_promedio=("temp_max", "mean"),
    precipitacion_total=("precipitacion", "sum"),
)
print(resumen)


In [ ]:
resumen = df.groupby("categoria").agg(
    dias=("categoria", "count"),
    temp_max_promedio=("temp_max", "mean"),
    precipitacion_total=("precipitacion", "sum"),
)
print(resumen)


### Paso 3.4 — Exportación de resultados

In [ ]:
%%writefile -a analisis.py
df.to_csv("pronostico_huancayo_procesado.csv", index=False)
resumen.to_csv("resumen_por_categoria.csv")
print("Archivos exportados.")


In [ ]:
df.to_csv("pronostico_huancayo_procesado.csv", index=False)
resumen.to_csv("resumen_por_categoria.csv")
print("Archivos exportados.")


### Paso 3.5 — Consolidación con Git
Actualice el README.md con una sección sobre el flujo de datos de esta semana antes de confirmar los cambios.

In [ ]:
!git add numpy_demo.py clima.py analisis.py *.csv *.json README.md requirements.txt
!git commit -m "feat: agrega pipeline de datos con NumPy, requests y Pandas (pronostico Huancayo)"
!git push origin main

## Entregable de la sesión
El mismo repositorio de GitHub de la Semana 1, actualizado con `numpy_demo.py`, `clima.py`, `analisis.py`, los cuatro archivos de datos generados, el README actualizado, y un commit nuevo en formato Conventional Commits.

## Checklist final rápido
- [ ] `numpy_demo.py` ejecuta sin errores y muestra la comparación bucle vs. vectorizado.
- [ ] `clima.py` consulta Open-Meteo con timeout y manejo de excepciones.
- [ ] Existen `pronostico_huancayo.json` y `pronostico_huancayo.csv` con 7 registros cada uno.
- [ ] `analisis.py` agrega las columnas derivadas y el groupby por categoría.
- [ ] Existen `pronostico_huancayo_procesado.csv` y `resumen_por_categoria.csv`.
- [ ] El README.md fue actualizado con la nueva sección de flujo de datos.
- [ ] El commit de esta semana sigue Conventional Commits y fue subido a GitHub.
